In [22]:
import astropy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os
from astropy.timeseries import LombScargle

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os
from astropy.timeseries import LombScargle
from toolz import itertoolz as tz
from scipy.stats import norm

# ===== YOUR PATHS =====
DATA_FOLDER = "/home/fernando-x390/Desktop/Astro with ML/Lab9/Be_GSEP"
OUTPUT_DIR  = "/home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output"

# Create output subfolders
os.makedirs(f"{OUTPUT_DIR}/individual_stars", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/summary_plots", exist_ok=True)

# ===== Publication-quality matplotlib settings =====
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.linewidth': 1.2,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'lines.linewidth': 1.5,
    'grid.alpha': 0.3,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

print(f"Data folder: {DATA_FOLDER}")
print(f"Output folder: {OUTPUT_DIR}")

folder_path = "/home/fernando-x390/Desktop/Astro with ML/Lab9/Be_GSEP/*.dat"
files = glob.glob(folder_path)

output_directory = "plots_output"
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

print(f"found{len(files)} files.")

for f in files:
    file_name = os.path.basename(f).replace(".dat", "")
    df = pd.read_csv(f,
                    sep=r'\s+', 
                    header=None, 
                    names=["Julian_Date", "Mag", "Mag_Error"],
                    usecols=[0, 1, 2])
    plt.figure(figsize=(15,4))
    plt.scatter(df["Julian_Date"], df["Mag"], s=5, alpha=0.6, label=file_name)
    plt.gca().invert_yaxis()  # Standard for Mag vs Time plots
    plt.xlabel('Julian Day')
    plt.ylabel('Magnitude (I/V)')
    plt.title(f'Light Curve: {file_name}')
    plt.legend()
    plt.savefig(f"{output_directory}/{file_name}.png")
    plt.close()


Data folder: /home/fernando-x390/Desktop/Astro with ML/Lab9/Be_GSEP
Output folder: /home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output
found50 files.


In [23]:
from toolz import itertoolz as tz
class FourierSeries:
    def __init__(self,n_terms=None,base_freq=None,custom_freqs=None):
        """Represents the b0 + Σₙ [ Aₙ cos(2πfₙt) + Bₙ sin(2πfₙt) ] series. 
        If initialized with only `n_terms`, the `base_freq` is assumed to be 1.0. 
        fₙ = n*base_freq, unless `custom_freqs` is provided, which is used to extract the fₙ.
        If both `custom_freqs` and `n_terms` are specified, the first n elements are used.
        If none are specified, the series will have one term by default.
        """
        self.fitted = False

        if custom_freqs is not None:
            if n_terms is None:
                n_terms = len(custom_freqs)
            self._freqs = np.array(custom_freqs[:n_terms])
        else:
            if n_terms is None:
                n_terms = 1
            if base_freq is None:
                base_freq = 1.0
            self._freqs = np.array([(i+1)*base_freq for i in range(n_terms)])
        self.n_terms = n_terms
        self._raw_coef = np.zeros(1+2*n_terms)
        self._param_names = ["base"] + list(tz.interleave((
            [f"A_{i+1}" for i in range(n_terms)],
            [f"B_{i+1}" for i in range(n_terms)]
        )))
    
    def __repr__(self):
        return f"<FourierSeries with {self.n_terms} terms>\nfitted: {self.fitted}\nparameters:\n" + self.parameters.to_string()
    
    def _design_matrix(self,ts):
        ts = np.array(ts)
        N = len(ts)
        const = np.ones(N)
        SIN = np.sin(2*np.pi*self._freqs[:,None]*ts[None,:])
        COS = np.cos(2*np.pi*self._freqs[:,None]*ts[None,:])
        return np.vstack([const] + list(tz.interleave([SIN,COS]))).T
        
    def __call__(self,t):
        scalar = np.isscalar(t)
        T = np.atleast_1d(t)
        X = self._design_matrix(T)
        return X @ self._raw_coef
    
    @property
    def parameters(self):
        return pd.Series(self._raw_coef,index=self._param_names)

    @property
    def A_n(self):
        return self._raw_coef[1::2]

    @property
    def B_n(self):
        return self._raw_coef[2::2]
        
        
    def fit(self,xdata,ydata,weights=None):
        """Performs OLS fit on the given data, which must be free of NaNs and infinite values.
        This instance is given the fitted parameters and the covariance matrix
        """
        X = self._design_matrix(xdata)
        self._raw_coef = np.linalg.pinv(X) @ ydata
        response = X @ self._raw_coef
        self.fitted = True
        return ydata - response

In [24]:
files = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.dat")))
print(f"Found {len(files)} .dat files in folder")

stars = []
for fpath in files:
    name = os.path.basename(fpath).replace(".dat", "")
    try:
        df = pd.read_csv(fpath, sep=r'\s+', header=None,
                         names=["JD", "Mag", "MagErr"], usecols=[0, 1, 2])
        # Clean
        mask = np.isfinite(df["JD"]) & np.isfinite(df["Mag"]) & np.isfinite(df["MagErr"])
        df = df[mask].sort_values("JD").reset_index(drop=True)
        
        if len(df) < 50:
            continue  # Skip stars with too few points
        
        stars.append({
            "name": name,
            "time": df["JD"].values,
            "mag": df["Mag"].values,
            "mag_err": df["MagErr"].values,
            "n": len(df)
        })
    except Exception as e:
        print(f"  ✗ Skipping {name}: {e}")

# Take first 50 valid stars
stars = stars[:50]
print(f"\n✓ Loaded {len(stars)} stars for analysis")
print(f"  Median # of points per star: {np.median([s['n'] for s in stars]):.0f}")

Found 50 .dat files in folder

✓ Loaded 50 stars for analysis
  Median # of points per star: 344


In [25]:
def find_period(time, mag, mag_err, p_min=1.0):
    baseline = time.max() - time.min()
    p_max   = baseline / 2.0
    f_min   = 1.0 / p_max
    f_max   = 1.0 / p_min
    df_step = 1.0 / (10 * baseline)  # frequency resolution
    
    freqs = np.arange(f_min, f_max, df_step)
    ls    = LombScargle(time, mag, mag_err)
    power = ls.power(freqs)
    
    best_idx  = np.argmax(power)
    best_freq = freqs[best_idx]
    best_per  = 1.0 / best_freq
    fap       = ls.false_alarm_probability(power[best_idx])
    return best_per, best_freq, freqs, power, fap

print("Finding periods...")
for i, s in enumerate(stars):
    p, f, freqs, power, fap = find_period(s["time"], s["mag"], s["mag_err"])
    s.update({"period": p, "freq": f, "freqs_arr": freqs,
              "power_arr": power, "fap": fap})
    print(f"  [{i+1:2d}/{len(stars)}] {s['name']:30s}  P = {p:8.3f} d   FAP = {fap:.2e}")

Finding periods...
  [ 1/50] LMC562.01.211                   P =    1.001 d   FAP = 1.45e-84
  [ 2/50] LMC562.01.7994                  P =    1.001 d   FAP = 1.71e-49
  [ 3/50] LMC562.02.7937                  P =    1.547 d   FAP = 4.49e-21
  [ 4/50] LMC562.02.8135                  P =    1.001 d   FAP = 3.14e-68
  [ 5/50] LMC562.03.8441                  P =  411.891 d   FAP = 8.12e-42
  [ 6/50] LMC562.04.125                   P =    1.001 d   FAP = 6.67e-33
  [ 7/50] LMC562.06.10895                 P =    1.001 d   FAP = 6.84e-84
  [ 8/50] LMC562.07.11068                 P =    1.001 d   FAP = 1.89e-47
  [ 9/50] LMC562.09.110                   P =    1.001 d   FAP = 2.07e-97
  [10/50] LMC562.11.87                    P =    1.001 d   FAP = 4.73e-31
  [11/50] LMC562.11.9588                  P =    1.002 d   FAP = 4.58e-09
  [12/50] LMC562.12.10123                 P =    1.001 d   FAP = 4.45e-27
  [13/50] LMC562.13.103                   P =    1.001 d   FAP = 1.95e-75
  [14/50] LMC562.13

In [26]:
N_FOURIER_TERMS = 7

def analyze_star(s, n_terms=N_FOURIER_TERMS):
    t, m, e, P = s["time"], s["mag"], s["mag_err"], s["period"]
    
    # Phase fold
    phase = ((t - t[0]) / P) % 1.0
    
    # Fourier fit at fundamental frequency
    fourier   = FourierSeries(n_terms=n_terms, base_freq=1.0/P)
    residuals = fourier.fit(t, m)
    
    # Statistics
    rms_orig = np.std(m)
    rms_res  = np.std(residuals)
    var_expl = 1 - (rms_res**2 / rms_orig**2)
    chi2_red = np.sum((residuals/e)**2) / (len(t) - (2*n_terms + 1))
    
    s.update({
        "phase": phase,
        "residuals": residuals,
        "fourier": fourier,
        "rms_orig": rms_orig,
        "rms_res": rms_res,
        "var_expl": var_expl,
        "chi2_red": chi2_red,
    })
    return s

print("\nPhase-folding and Fourier-fitting...")
for i, s in enumerate(stars):
    analyze_star(s)
    print(f"  [{i+1:2d}/{len(stars)}] {s['name']:30s}  "
          f"σ_res = {s['rms_res']:.4f} mag   Var.Expl = {s['var_expl']*100:5.1f}%")


Phase-folding and Fourier-fitting...
  [ 1/50] LMC562.01.211                   σ_res = 0.0359 mag   Var.Expl =  84.4%
  [ 2/50] LMC562.01.7994                  σ_res = 0.1199 mag   Var.Expl =  63.0%
  [ 3/50] LMC562.02.7937                  σ_res = 0.0184 mag   Var.Expl =  31.2%
  [ 4/50] LMC562.02.8135                  σ_res = 0.0310 mag   Var.Expl =  67.0%
  [ 5/50] LMC562.03.8441                  σ_res = 0.0388 mag   Var.Expl =  65.8%
  [ 6/50] LMC562.04.125                   σ_res = 0.0253 mag   Var.Expl =  40.8%
  [ 7/50] LMC562.06.10895                 σ_res = 0.0479 mag   Var.Expl =  70.5%
  [ 8/50] LMC562.07.11068                 σ_res = 0.0381 mag   Var.Expl =  57.0%
  [ 9/50] LMC562.09.110                   σ_res = 0.0428 mag   Var.Expl =  77.7%
  [10/50] LMC562.11.87                    σ_res = 0.0354 mag   Var.Expl =  50.4%
  [11/50] LMC562.11.9588                  σ_res = 0.0405 mag   Var.Expl =  22.5%
  [12/50] LMC562.12.10123                 σ_res = 0.0244 mag   Var.Expl

In [30]:
def plot_single_star(s, save=True):
    name, P = s["name"], s["period"]
    t, m, e = s["time"], s["mag"], s["mag_err"]
    phase, res, fourier = s["phase"], s["residuals"], s["fourier"]
    
    # Smooth model curve in phase
    p_smooth = np.linspace(0, 1, 500)
    t_smooth = p_smooth * P + t[0]
    m_smooth = fourier(t_smooth)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(f"Be Star {name}  |  P = {P:.3f} d  |  N = {s['n']}",
                 fontsize=14, fontweight='bold')
    
    # (1) Periodogram
    ax = axes[0, 0]
    ax.plot(1.0/s["freqs_arr"], s["power_arr"], 'k-', lw=0.8)
    ax.axvline(P, color='red', ls='--', lw=1.5, label=f'P = {P:.3f} d')
    ax.set_xscale('log')
    ax.set_xlabel('Period (days)'); ax.set_ylabel('LS Power')
    ax.set_title('Lomb-Scargle Periodogram'); ax.legend(); ax.grid(alpha=0.3)
    
    # (2) Phase-folded curve + Fourier
    ax = axes[0, 1]
    ax.errorbar(phase, m, yerr=e, fmt='o', ms=3, color='gray',
                alpha=0.5, elinewidth=0.4, capsize=0, label='Data')
    ax.plot(p_smooth, m_smooth, 'r-', lw=2.2, label=f'Fourier ({N_FOURIER_TERMS} terms)')
    ax.invert_yaxis()
    ax.set_xlabel('Phase'); ax.set_ylabel('Magnitude')
    ax.set_title(f'Phase-folded  (Var.Expl = {s["var_expl"]*100:.1f}%)')
    ax.legend(); ax.grid(alpha=0.3)
    
    # (3) Residuals vs phase
    ax = axes[1, 0]
    sigma = s["rms_res"]
    ax.scatter(phase, res, s=10, alpha=0.5, c='navy')
    ax.axhline(0, color='red', lw=1.5)
    ax.axhline( sigma, color='orange', ls='--', lw=1.2, label=f'±1σ = {sigma:.4f}')
    ax.axhline(-sigma, color='orange', ls='--', lw=1.2)
    ax.set_xlabel('Phase'); ax.set_ylabel('Residuals (mag)')
    ax.set_title('Non-periodic Variability (Be-star dispersion)')
    ax.legend(); ax.grid(alpha=0.3)
    
    # (4) Residuals histogram + Gaussian
    ax = axes[1, 1]
    ax.hist(res, bins=30, density=True, color='steelblue',
            edgecolor='black', alpha=0.7)
    mu, sg = norm.fit(res)
    x = np.linspace(res.min(), res.max(), 200)
    ax.plot(x, norm.pdf(x, mu, sg), 'k-', lw=2,
            label=f'Gaussian: μ={mu:.4f}, σ={sg:.4f}')
    ax.axvline(0, color='red', ls='--', lw=1.5)
    ax.set_xlabel('Residual (mag)'); ax.set_ylabel('Density')
    ax.set_title(f'Residual Distribution  (χ²ᵣ = {s["chi2_red"]:.2f})')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    
    plt.tight_layout()
    if save:
        out = f"{OUTPUT_DIR}/individual_stars/{name}.png"
        plt.savefig(out, dpi=300)
    plt.close()

In [31]:
print("\nGenerating individual star plots...")
for i, s in enumerate(stars):
    plot_single_star(s, save=True)
    if (i+1) % 10 == 0:
        print(f"  ✓ {i+1}/{len(stars)} plots saved")
print(f"\n✓ All individual plots in: {OUTPUT_DIR}/individual_stars/")


Generating individual star plots...
  ✓ 10/50 plots saved
  ✓ 20/50 plots saved
  ✓ 30/50 plots saved
  ✓ 40/50 plots saved
  ✓ 50/50 plots saved

✓ All individual plots in: /home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output/individual_stars/


In [32]:
summary = pd.DataFrame([{
    "Star":          s["name"],
    "Period_d":      s["period"],
    "N_points":      s["n"],
    "RMS_orig":      s["rms_orig"],
    "RMS_res":       s["rms_res"],
    "Var_Explained": s["var_expl"],
    "Chi2_red":      s["chi2_red"],
    "FAP":           s["fap"],
} for s in stars])

summary_csv = f"{OUTPUT_DIR}/summary_statistics.csv"
summary.to_csv(summary_csv, index=False, float_format='%.6f')
print(f"\n✓ Summary saved to {summary_csv}")
print(summary.describe())


✓ Summary saved to /home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output/summary_statistics.csv
         Period_d    N_points   RMS_orig    RMS_res  Var_Explained  \
count   50.000000   50.000000  50.000000  50.000000      50.000000   
mean    25.236360  340.860000   0.064578   0.039842       0.570693   
std     80.918721    7.529561   0.034973   0.020685       0.168797   
min      1.000342  312.000000   0.022161   0.014262       0.154810   
25%      1.000828  338.250000   0.037869   0.024607       0.444200   
50%      1.001066  344.000000   0.057319   0.035905       0.605945   
75%      1.001527  345.000000   0.085941   0.045894       0.715125   
max    411.890640  345.000000   0.197148   0.119888       0.844776   

         Chi2_red            FAP  
count   50.000000   5.000000e+01  
mean    53.995704   1.328900e-09  
std     96.717465   8.643375e-09  
min      2.891276  6.479337e-103  
25%      9.080942   9.134625e-75  
50%     22.342281   1.009521e-47  
75%     53.988690 

In [33]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Be Stars Sample Statistics (N = {})'.format(len(summary)),
             fontsize=15, fontweight='bold')

# Period distribution
ax = axes[0, 0]
ax.hist(summary["Period_d"], bins=20, color='steelblue', edgecolor='black', alpha=0.8)
ax.axvline(summary["Period_d"].median(), color='red', ls='--', lw=2,
           label=f'Median = {summary["Period_d"].median():.2f} d')
ax.set_xlabel('Period (days)'); ax.set_ylabel('# stars')
ax.set_title('Period Distribution'); ax.legend(); ax.grid(alpha=0.3)

# Variance explained
ax = axes[0, 1]
ax.hist(summary["Var_Explained"]*100, bins=20, color='forestgreen',
        edgecolor='black', alpha=0.8)
ax.axvline(summary["Var_Explained"].median()*100, color='red', ls='--', lw=2,
           label=f'Median = {summary["Var_Explained"].median()*100:.1f}%')
ax.set_xlabel('Variance Explained (%)'); ax.set_ylabel('# stars')
ax.set_title('Strength of Periodic Component'); ax.legend(); ax.grid(alpha=0.3)

# RMS residuals
ax = axes[1, 0]
ax.hist(summary["RMS_res"], bins=20, color='coral', edgecolor='black', alpha=0.8)
ax.axvline(summary["RMS_res"].median(), color='red', ls='--', lw=2,
           label=f'Median = {summary["RMS_res"].median():.4f} mag')
ax.set_xlabel('RMS of Residuals (mag)'); ax.set_ylabel('# stars')
ax.set_title('Non-periodic Dispersion'); ax.legend(); ax.grid(alpha=0.3)

# Period vs RMS residuals (color = var_expl)
ax = axes[1, 1]
sc = ax.scatter(summary["Period_d"], summary["RMS_res"],
                c=summary["Var_Explained"]*100, cmap='viridis',
                s=70, edgecolor='black', alpha=0.8)
plt.colorbar(sc, ax=ax, label='Variance Explained (%)')
ax.set_xscale('log')
ax.set_xlabel('Period (days)'); ax.set_ylabel('RMS Residuals (mag)')
ax.set_title('Dispersion vs Period'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/summary_plots/sample_statistics.png", dpi=300)
plt.close()
print(f"✓ Sample statistics plot saved")

✓ Sample statistics plot saved


In [34]:
top_n = 10
top_stars = summary.sort_values("RMS_res", ascending=False).head(top_n)
top_names = top_stars["Star"].tolist()
top_results = [s for s in stars if s["name"] in top_names]
top_results = sorted(top_results, key=lambda x: -x["rms_res"])

fig, axes = plt.subplots(5, 2, figsize=(15, 18))
fig.suptitle(f'Top {top_n} Be Stars: Highest Non-periodic Dispersion',
             fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, s in enumerate(top_results):
    ax = axes[i]
    ax.scatter(s["phase"], s["residuals"], s=12, alpha=0.5, c='navy')
    sigma = s["rms_res"]
    ax.axhline(0, color='red', lw=1.5)
    ax.axhline( sigma, color='orange', ls='--', lw=1.2)
    ax.axhline(-sigma, color='orange', ls='--', lw=1.2)
    ax.set_xlabel('Phase'); ax.set_ylabel('Residuals (mag)')
    ax.set_title(f"{s['name']}  σ={sigma:.4f} mag  P={s['period']:.2f} d", fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/summary_plots/top_dispersed_stars.png", dpi=300)
plt.close()
print(f"✓ Top {top_n} dispersed stars plot saved")

✓ Top 10 dispersed stars plot saved


In [35]:
report_path = f"{OUTPUT_DIR}/ANALYSIS_REPORT.txt"
with open(report_path, 'w') as f:
    f.write("="*72 + "\n")
    f.write("  BE STARS PHOTOMETRIC VARIABILITY — ANALYSIS REPORT\n")
    f.write("="*72 + "\n\n")
    f.write(f"Generated: {pd.Timestamp.now():%Y-%m-%d %H:%M:%S}\n")
    f.write(f"Stars analyzed: {len(summary)}\n")
    f.write(f"Fourier terms: {N_FOURIER_TERMS}\n\n")
    
    f.write("--- Periods ---\n")
    f.write(f"  Range:   {summary['Period_d'].min():.3f} – {summary['Period_d'].max():.3f} d\n")
    f.write(f"  Median:  {summary['Period_d'].median():.3f} d\n\n")
    
    f.write("--- Periodic component (variance explained) ---\n")
    f.write(f"  Mean:    {summary['Var_Explained'].mean()*100:.1f}%\n")
    f.write(f"  Median:  {summary['Var_Explained'].median()*100:.1f}%\n")
    f.write(f"  > 50%:   {(summary['Var_Explained']>0.5).sum()} stars\n\n")
    
    f.write("--- Non-periodic dispersion (RMS of residuals) ---\n")
    f.write(f"  Mean:    {summary['RMS_res'].mean():.4f} mag\n")
    f.write(f"  Median:  {summary['RMS_res'].median():.4f} mag\n")
    f.write(f"  Range:   {summary['RMS_res'].min():.4f} – {summary['RMS_res'].max():.4f} mag\n\n")
    
    f.write("--- Goodness of fit ---\n")
    f.write(f"  Median χ²ᵣ:  {summary['Chi2_red'].median():.2f}\n\n")
    
    f.write("--- Top 10 most dispersed (Be-star-like) stars ---\n")
    f.write(f"{'Rank':<5}{'Star':<32}{'σ_res (mag)':<14}{'Var.Expl (%)':<14}{'P (d)':<10}\n")
    f.write("-"*72 + "\n")
    for i, (_, row) in enumerate(top_stars.iterrows(), 1):
        f.write(f"{i:<5}{row['Star']:<32}{row['RMS_res']:<14.4f}"
                f"{row['Var_Explained']*100:<14.1f}{row['Period_d']:<10.2f}\n")
    f.write("\n" + "="*72 + "\n")

print(f"\n✓ Report saved: {report_path}")
print("\n" + "="*60)
print("  ANALYSIS COMPLETE")
print("="*60)
print(f"  • Individual plots:  {OUTPUT_DIR}/individual_stars/")
print(f"  • Summary plots:     {OUTPUT_DIR}/summary_plots/")
print(f"  • CSV table:         {OUTPUT_DIR}/summary_statistics.csv")
print(f"  • Text report:       {OUTPUT_DIR}/ANALYSIS_REPORT.txt")


✓ Report saved: /home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output/ANALYSIS_REPORT.txt

  ANALYSIS COMPLETE
  • Individual plots:  /home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output/individual_stars/
  • Summary plots:     /home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output/summary_plots/
  • CSV table:         /home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output/summary_statistics.csv
  • Text report:       /home/fernando-x390/Desktop/Astro with ML/Lab9/be_stars_output/ANALYSIS_REPORT.txt
